**Install Dependencies & Import Modules**

In [0]:
# %pip install pydantic

import os
import time
import requests
import json
from datetime import datetime
from decimal import Decimal, InvalidOperation
from typing import Optional, List, Dict, Any, Tuple
from pydantic import BaseModel, Field, field_validator
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType


**Define Configuration and Safe Environment Secrets**

In [0]:
# Interactive UI Widgets entered by the developer running the notebook
BASE_URL = os.environ.get("ASSESSMENT_API_BASE_URL") or dbutils.widgets.get("assessment_api_base_url")
API_KEY = os.environ.get("ASSESSMENT_API_KEY") or dbutils.widgets.get("assessment_api_key")
AUTH_TOKEN = os.environ.get("ASSESSMENT_AUTH_TOKEN") or dbutils.widgets.get("assessment_auth_token")

# If keys are missing, raise a clear operational error instead of failing with a generic 401/403 down the line.
if not BASE_URL or not API_KEY or not AUTH_TOKEN:
    raise ValueError(
        "CRITICAL SECURITY CONFIGURATION ERROR:\n"
        "Required API credentials are missing. Please either set the corresponding environment variables "
        "or fill out the parameters in the Databricks Widget text boxes at the top of your screen."
    )
# Unity Catalog table and volume locations
BRONZE_TABLE = "workspace.default.bronze_transactions"
QUARANTINE_TABLE = "workspace.default.quarantine_transactions"
WATERMARK_FILE_PATH = "/Volumes/workspace/default/ingestion_state/watermark.json"

# Currencies and Categories Enums provided by the Transaction Schema rules
VALID_CURRENCIES = {"USD", "EUR", "GBP", "CHF", "JPY", "AUD", "CAD"}
VALID_CATEGORIES = {
    "e-commerce", "travel", "food_and_beverage", "groceries", "electronics", 
    "retail", "entertainment", "health", "transportation", "home_and_garden", 
    "payroll", "transfer"
}
VALID_STATUSES = {"completed", "pending", "failed", "reversed"}


**Setup Schema Validation via Pydantic**

In [0]:
class TransactionModel(BaseModel):
    transaction_id: str = Field(..., min_length=5)
    account_id: str = Field(..., min_length=5)
    transaction_date: str
    amount: float
    currency: str
    transaction_type: str
    merchant_name: str
    merchant_category: str
    status: str
    country_code: str

    @field_validator('amount')
    def validate_amount(cls, v):
        if v <= 0:
            raise ValueError("Amount must be strictly greater than zero.")
        return v

    @field_validator('currency')
    def validate_currency(cls, v):
        if v not in VALID_CURRENCIES:
            raise ValueError(f"Invalid currency code: {v}")
        return v

    @field_validator('transaction_type')
    def validate_type(cls, v):
        if v not in {"debit", "credit"}:
            raise ValueError("Transaction type must be lowercase 'debit' or 'credit'.")
        return v

    @field_validator('merchant_name')
    def validate_merchant(cls, v):
        if not v or not v.strip():
            raise ValueError("Merchant name cannot be empty or whitespace-only.")
        return v

    @field_validator('merchant_category')
    def validate_category(cls, v):
        if v not in VALID_CATEGORIES:
            raise ValueError(f"Invalid merchant category: {v}")
        return v

    @field_validator('status')
    def validate_status(cls, v):
        if v not in VALID_STATUSES:
            raise ValueError("Status must be lowercase completed, pending, failed, or reversed.")
        return v

    @field_validator('country_code')
    def validate_country(cls, v):
        if len(v) != 2 or not v.isupper():
            raise ValueError("Country code must be assigned ISO 3166-1 alpha-2 format.")
        return v


**API Pagination, Watermarking, and Exponential Retry Logic**

In [0]:
def fetch_transactions_page(limit: int, offset: int, watermark_date: str) -> List[Dict[str, Any]]:
    """Fetches data from the API endpoint using exponential backoff retry mechanics."""
    url = f"{BASE_URL}/transactions"
    headers = {
        "apikey": API_KEY,
        "Authorization": f"Bearer {AUTH_TOKEN}"
    }
    params = {
        "limit": limit,
        "offset": offset,
        "transaction_date": f"gte.{watermark_date}",
        "order": "transaction_date.asc"
    }
    
    retries = 3
    backoff = 2
    
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, params=params, timeout=15)
            if response.status_code == 200:
                return response.json()
            elif response.status_code in (429, 500, 502, 503, 504):
                print(f"Temporary server error code {response.status_code}. Retrying in {backoff} seconds...")
                time.sleep(backoff)
                backoff *= 2
            else:
                print(f"Critical execution failure. Status Code: {response.status_code}, Msg: {response.text}")
                return []
        except requests.exceptions.RequestException as e:
            print(f"Network exception encountered: {e}. Retrying...")
            time.sleep(backoff)
            backoff *= 2
            
    return []

def get_last_watermark() -> str:
    """Reads the previous processing session's high-watermark timestamp."""
    try:
        if os.path.exists(WATERMARK_FILE_PATH):
            with open(WATERMARK_FILE_PATH, 'r') as f:
                return json.load(f).get("max_transaction_date", "2024-01-01T00:00:00Z")
    except Exception as e:
        print(f"Could not load watermark metadata: {e}. Defaulting baseline initialization.")
    return "2024-01-01T00:00:00Z"

def save_new_watermark(new_timestamp: str):
    """Saves the current run's high-watermark timestamp checkpoint to a UC volume."""
    os.makedirs(os.path.dirname(WATERMARK_FILE_PATH), exist_ok=True)
    with open(WATERMARK_FILE_PATH, 'w') as f:
        json.dump({"max_transaction_date": new_timestamp, "processed_at": datetime.utcnow().isoformat() + "Z"}, f)


**Main Orchestration Engine**

In [0]:
def run_ingestion_pipeline():
    watermark = get_last_watermark()
    print(f"Initiating ingestion cycle utilizing high-watermark threshold: {watermark}")
    
    limit = 100
    offset = 0
    all_raw_records = []
    
    # Loop over API offsets iteratively
    while True:
        records = fetch_transactions_page(limit, offset, watermark)
        if not records:
            break
        all_raw_records.extend(records)
        if len(records) < limit:
            break
        offset += limit
        
    print(f"Acquired total payload stack containing {len(all_raw_records)} transactions from source endpoint.")
    
    valid_rows = []
    quarantine_rows = []
    max_date_tracked = watermark
    natural_keys_seen = set()
    
    ingestion_time = datetime.utcnow()
    
    schema_field_names = ["transaction_id", "account_id", "transaction_date", "amount", "currency", "transaction_type", "merchant_name", "merchant_category", "status", "country_code"]
    
    for rec in all_raw_records:
        # Task 3 tracking: Capture latest valid timestamp string evaluated
        tx_date = rec.get("transaction_date", "")
        if tx_date > max_date_tracked:
            max_date_tracked = tx_date
            
        # Deduplication implementation (Task 1): Check natural key uniqueness
        # Natural Key = Combines all transaction specific details except transaction_id
        natural_key = (
            rec.get("account_id"), rec.get("transaction_date"), str(rec.get("amount")),
            rec.get("currency"), rec.get("transaction_type"), rec.get("merchant_name"),
            rec.get("merchant_category"), rec.get("status"), rec.get("country_code")
        )
        
        if natural_key in natural_keys_seen:
            # Route duplicate entity instances directly to quarantine dead-letter table
            rec["error_reason"] = "Duplicate transaction record detected based on structural natural key validation rules."
            rec["ingestion_timestamp"] = ingestion_time
            quarantine_rows.append(Row(**{k: rec.get(k) for k in schema_field_names}, ingestion_timestamp=ingestion_time, error_reason=rec.get("error_reason", "")))
            continue
            
        try:
            # Validate schema layout requirements using Pydantic structure
            TransactionModel(**rec)
            
            # Form clean row structure for Bronze storage
            natural_keys_seen.add(natural_key)
            rec["ingestion_timestamp"] = ingestion_time
            valid_rows.append(Row(**{k: rec.get(k) for k in schema_field_names}, ingestion_timestamp=ingestion_time))
            
        except Exception as err_details:
            # Handle schema violations cleanly without breaking execution loops
            rec["error_reason"] = str(err_details)
            rec["ingestion_timestamp"] = ingestion_time
            quarantine_rows.append(Row(**{k: rec.get(k) for k in schema_field_names}, ingestion_timestamp=ingestion_time, error_reason=rec.get("error_reason", "")))

    # Define strict Target Data Spark Schemas
    schema_base = [
        StructField("transaction_id", StringType(), True),
        StructField("account_id", StringType(), True),
        StructField("transaction_date", StringType(), True),
        StructField("amount", StringType(), True), # Stored as string or decimal for precision parsing
        StructField("currency", StringType(), True),
        StructField("transaction_type", StringType(), True),
        StructField("merchant_name", StringType(), True),
        StructField("merchant_category", StringType(), True),
        StructField("status", StringType(), True),
        StructField("country_code", StringType(), True),
        StructField("ingestion_timestamp", TimestampType(), True)
    ]
    
    # Save valid rows to Bronze Layer
    if valid_rows:
        df_valid = spark.createDataFrame(valid_rows, StructType(schema_base))
        df_valid.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
        print(f"Successfully committed {len(valid_rows)} verified records to Bronze Lakehouse storage layer.")
        
    # Save defective rows to Quarantine Layer
    if quarantine_rows:
        schema_quarantine = StructType(schema_base + [StructField("error_reason", StringType(), True)])
        df_quarantine = spark.createDataFrame(quarantine_rows, schema_quarantine)
        df_quarantine.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)
        print(f"Quarantined {len(quarantine_rows)} defective entries inside dead-letter collection.")
        
        # Output visual file results for assessment checking
        display(df_quarantine.limit(10))

    # Checkpoint watermark progress safely 
    if max_date_tracked > watermark:
        save_new_watermark(max_date_tracked)
        
run_ingestion_pipeline()
